In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

neo4j_uri = (os.getenv("NEO4J_URI") or "").strip().strip('"').strip("'")
if "daases.neo4j.io" in neo4j_uri:
    neo4j_uri = neo4j_uri.replace("daases.neo4j.io", "databases.neo4j.io")
os.environ["NEO4J_URI"] = neo4j_uri
os.environ["NEO4J_USERNAME"] = os.getenv("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = os.getenv("NEO4J_PASSWORD")
print("NEO4J_URI:", os.environ["NEO4J_URI"])

NEO4J_URI: neo4j+s://1e06e23e.databases.neo4j.io


In [3]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=os.getenv("NEO4J_URI"),username=os.getenv("NEO4J_USERNAME"),password=os.getenv("NEO4J_PASSWORD"))
graph

C:\Users\harry\AppData\Local\Temp\ipykernel_5336\2830426522.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=os.getenv("NEO4J_URI"),username=os.getenv("NEO4J_USERNAME"),password=os.getenv("NEO4J_PASSWORD"))


In [4]:
## Dataset Moview 
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [5]:
try:
    graph.query(moview_query)
except Exception as e:
    print(f"LOAD CSV failed: {e}")
    fallback_query = """
    MERGE (c:Movie {id:'casino_1995'})
    SET c.title='Casino', c.imdbRating=8.2, c.released=date('1995-11-22')
    MERGE (sc:Person {name:'Martin Scorsese'})
    MERGE (sc)-[:DIRECTED]->(c)
    FOREACH (actor IN ['Robert De Niro','Sharon Stone','Joe Pesci'] |
      MERGE (p:Person {name:actor})
      MERGE (p)-[:ACTED_IN]->(c)
    )

    MERGE (f:Movie {id:'forrest_gump_1994'})
    SET f.title='Forrest Gump', f.imdbRating=8.8, f.released=date('1994-07-06')
    MERGE (rz1:Person {name:'Robert Zemeckis'})
    MERGE (rz1)-[:DIRECTED]->(f)
    FOREACH (actor IN ['Tom Hanks','Robin Wright','Gary Sinise'] |
      MERGE (p:Person {name:actor})
      MERGE (p)-[:ACTED_IN]->(f)
    )

    MERGE (ca:Movie {id:'cast_away_2000'})
    SET ca.title='Cast Away', ca.imdbRating=7.8, ca.released=date('2000-12-22')
    MERGE (rz2:Person {name:'Robert Zemeckis'})
    MERGE (rz2)-[:DIRECTED]->(ca)
    FOREACH (actor IN ['Tom Hanks','Helen Hunt'] |
      MERGE (p:Person {name:actor})
      MERGE (p)-[:ACTED_IN]->(ca)
    )
    """
    graph.query(fallback_query)

movie_count = graph.query("MATCH (m:Movie) RETURN count(m) AS movie_count")[0]["movie_count"]
graph.refresh_schema()
print("Movie count:", movie_count)
print(graph.schema)

Movie count: 299
Node properties:
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {postId: INTEGER, content: STRING, timestamp: DATE_TIME}
Movie {id: STRING, title: STRING, released: DATE, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:User)-[:POSTED]->(:Post)
(:User)-[:FRIEND]->(:User)
(:User)-[:LIKES]->(:User)
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [6]:
graph.refresh_schema()
print(graph.schema)

Node properties:
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {postId: INTEGER, content: STRING, timestamp: DATE_TIME}
Movie {id: STRING, title: STRING, released: DATE, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:User)-[:POSTED]->(:Post)
(:User)-[:FRIEND]->(:User)
(:User)-[:LIKES]->(:User)
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [8]:
## Initialize LLM model 
from langchain_groq import ChatGroq
llm = ChatGroq(model="meta-llama/llama-4-maverick-17b-128e-instruct", temperature=0)
llm 
 

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002463227B5C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000246257DE720>, model_name='meta-llama/llama-4-maverick-17b-128e-instruct', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
graph.query("MATCH (m:Movie) RETURN count(m) AS movie_count")
print(graph.schema)

Node properties:
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {postId: INTEGER, content: STRING, timestamp: DATE_TIME}
Movie {id: STRING, title: STRING, released: DATE, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:User)-[:POSTED]->(:Post)
(:User)-[:FRIEND]->(:User)
(:User)-[:LIKES]->(:User)
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [10]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

def ask_graph(question: str):
    graph.refresh_schema()
    movie_rows = graph.query("MATCH (m:Movie) RETURN count(m) AS movie_count")
    movie_count = movie_rows[0]["movie_count"] if movie_rows else 0
    if movie_count == 0:
        raise ValueError("No Movie nodes found. Run the movie data load cell first.")

    local_chain = GraphCypherQAChain.from_llm(
        graph=graph,
        llm=llm,
        verbose=True,
        allow_dangerous_requests=True,
    )
    return local_chain.invoke({"query": question})

In [11]:
response = ask_graph("Who was the director of the movie Casino")
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'}) RETURN p.name
Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


{'query': 'Who was the director of the movie Casino',
 'result': 'Martin Scorsese.'}

In [12]:
response = ask_graph("Who were the actors of the movie Casino")
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Casino'}) RETURN p.name
Full Context:
[{'p.name': 'Robert De Niro'}, {'p.name': 'Joe Pesci'}, {'p.name': 'Sharon Stone'}, {'p.name': 'James Woods'}]

> Finished chain.


{'query': 'Who were the actors of the movie Casino',
 'result': 'Robert De Niro, Joe Pesci, Sharon Stone, James Woods were the actors of the movie Casino.'}

In [13]:
response = ask_graph("How many artists are there?")
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person) RETURN count(p)
Full Context:
[{'count(p)': 1239}]

> Finished chain.


{'query': 'How many artists are there?', 'result': 'There are 1239 artists.'}

In [14]:
response = ask_graph("How many movies has Tom Hanks acted in")
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)
Full Context:
[{'count(m)': 2}]

> Finished chain.


{'query': 'How many movies has Tom Hanks acted in',
 'result': 'Tom Hanks has acted in 2 movies.'}